# 06 · Tracking multiobjeto (MOT) — el corazón del puesto (≈4 h)

La oferta: *"detección y seguimiento simultáneo de múltiples personas"*, *"conservación de identidades durante
cruces, oclusiones y desapariciones"*, *"tracking multiobjeto y asociación temporal de detecciones"*,
*"evaluación y comparación de estrategias de tracking"*, *"análisis de cambios incorrectos de identidad"*, y
valoran *ByteTrack, BoT-SORT, OC-SORT, DeepSORT*.

Plan:
1. El problema y el paradigma *tracking-by-detection*
2. Filtro de Kalman ✍️
3. Asociación con el algoritmo húngaro
4. SORT desde cero ✍️
5. Métricas: MOTA, IDF1, HOTA (y por qué discrepan)
6. ByteTrack y ajuste de hiperparámetros
7. Análisis de ID switches
8. Post-procesado offline: interpolación ✍️
9. Mapa de trackers modernos (DeepSORT, OC-SORT, BoT-SORT…)
10. Tracking real con Ultralytics sobre vídeo

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte, clona el repo y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
if ROOT is None:   # este notebook usa código del repo (vision_utils/ o 11_produccion/) → clonarlo
    ROOT = Path.cwd() / "vision-bootcamp"
    url = "https://github.com/jcandel01/vision-bootcamp.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(ROOT)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo: " + r.stderr)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

from vision_utils.synthetic import make_scene, draw_scene
from vision_utils.mot_metrics import evaluate, iou_matrix
from vision_utils.tracker import Tracker, run_tracker
pd.set_option("display.width", 200)

## 1. El problema

Entrada: vídeo. Salida: para cada frame, cajas **con un ID persistente** por objeto. Formato MOT:
`frame, id, x, y, w, h`.

**Tracking-by-detection** (el paradigma dominante y el que usan todos los trackers de la oferta):
```
frame t ─► DETECTOR ─► detecciones D_t ─┐
                                         ├─► ASOCIACIÓN (coste + húngaro) ─► actualizar / crear / borrar tracks
tracks T_{t-1} ─► MODELO DE MOVIMIENTO ──┘          ▲
                  (Kalman: predice dónde estarán)   └── opcional: APARIENCIA (ReID), compensación de cámara
```
Tres piezas: **modelo de movimiento** (dónde espero a cada track), **asociación** (qué detección es de quién) y
**gestión del ciclo de vida** (cuándo nace, se confirma, se pierde y muere un track).

Dificultades (todas aparecen en deporte):
- **Oclusiones** y **cruces**: dos jugadores se solapan → el detector da una caja o ninguna → riesgo de
  intercambio de IDs.
- **Apariencia similar**: mismo uniforme → la apariencia ayuda poco (DanceTrack y SportsMOT existen por esto).
- **Movimiento no lineal**: cambios bruscos de dirección → el Kalman de velocidad constante se equivoca.
- **Cámara en movimiento** (paneo/zoom) → todo se mueve en la imagen aunque los jugadores estén quietos.
- **Entradas/salidas** del encuadre y desapariciones largas.

## 2. Filtro de Kalman

Estima el estado oculto de un sistema a partir de medidas ruidosas, alternando dos pasos:

**Predict** (modelo de movimiento): $\hat x = F x$, $\;P = F P F^\top + Q$
→ "si iba a esta velocidad, ahora estará aquí"; la incertidumbre $P$ **crece** (ruido de proceso $Q$).

**Update** (cuando llega una detección $z$):
- innovación $y = z - H\hat x$ (cuánto se equivocó la predicción)
- $S = H P H^\top + R$ (incertidumbre de esa innovación; $R$ = ruido del detector)
- ganancia $K = P H^\top S^{-1}$ → cuánto me fío de la medida frente a la predicción
- $x = \hat x + K y$, $\;P = (I - K H) P$ → la incertidumbre **baja**.

En tracking: estado $x = [c_x, c_y, w, h, v_x, v_y, v_w, v_h]$ (BoT-SORT; SORT usa área y aspecto,
ByteTrack aspecto y altura), modelo de **velocidad constante**:
$F = \begin{bmatrix} I_4 & I_4 \\ 0 & I_4 \end{bmatrix}$, y observamos solo la caja: $H = [I_4 \;\; 0]$.

**Por qué es clave**: durante una oclusión no hay detección → solo se hace *predict*, el track "sigue avanzando"
y cuando la persona reaparece la caja predicha está cerca → se re-asocia con el mismo ID.

✍️ **EJERCICIO**: completa `predict` y `update`.

In [ ]:
class MyKalmanBox:
    std_pos, std_vel = 1 / 20, 1 / 160

    def __init__(self, z):                       # z = [cx, cy, w, h]
        z = np.asarray(z, float)
        self.F = np.eye(8); self.F[:4, 4:] = np.eye(4)
        self.H = np.eye(4, 8)
        self.x = np.r_[z, np.zeros(4)]
        h = z[3]
        self.P = np.diag(np.r_[np.full(4, (2 * self.std_pos * h) ** 2), np.full(4, (10 * self.std_vel * h) ** 2)])

    def Q(self):
        h = self.x[3]
        return np.diag(np.r_[np.full(4, (self.std_pos * h) ** 2), np.full(4, (self.std_vel * h) ** 2)])

    def R(self):
        return np.diag(np.full(4, (self.std_pos * self.x[3]) ** 2))

    def predict(self):
        # TU CÓDIGO AQUÍ: actualiza self.x y self.P; devuelve self.x[:4]
        raise NotImplementedError

    def update(self, z):
        # TU CÓDIGO AQUÍ
        raise NotImplementedError

In [ ]:
# SOLUCIÓN
def _predict(self):
    self.x = self.F @ self.x
    self.P = self.F @ self.P @ self.F.T + self.Q()
    return self.x[:4]

def _update(self, z):
    S = self.H @ self.P @ self.H.T + self.R()
    K = self.P @ self.H.T @ np.linalg.inv(S)
    self.x = self.x + K @ (np.asarray(z, float) - self.H @ self.x)
    self.P = (np.eye(8) - K @ self.H) @ self.P

MyKalmanBox.predict, MyKalmanBox.update = _predict, _update

In [ ]:
# Prueba: una persona que camina en línea recta, detector ruidoso y una oclusión entre los frames 40 y 60
rng = np.random.default_rng(0)
T = 100
true = np.c_[100 + 4 * np.arange(T), 300 + 1.5 * np.arange(T), np.full(T, 40), np.full(T, 100)]
meas = true + rng.normal(0, 4, true.shape)
kf = MyKalmanBox(meas[0])
est, unc = [kf.x[:4].copy()], [np.sqrt(kf.P[0, 0])]
for t in range(1, T):
    kf.predict()
    if not (40 <= t < 60):
        kf.update(meas[t])
    est.append(kf.x[:4].copy()); unc.append(np.sqrt(kf.P[0, 0]))
est, unc = np.array(est), np.array(unc)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(true[:, 0], true[:, 1], "k-", label="real")
ax[0].plot(meas[:, 0], meas[:, 1], ".", alpha=0.4, label="detecciones")
ax[0].plot(est[:, 0], est[:, 1], "r-", label="Kalman")
ax[0].axvspan(true[40, 0], true[60, 0], color="orange", alpha=0.2, label="oclusión"); ax[0].legend(); ax[0].invert_yaxis()
ax[1].plot(unc); ax[1].axvspan(40, 60, color="orange", alpha=0.2); ax[1].set_title("σ de cx: crece sin medidas, cae al volver")
plt.show()
print(f"error medio con detecciones: {np.abs(meas - true)[:, 0].mean():.2f}px | Kalman: {np.abs(est - true)[:, 0].mean():.2f}px")
print(f"error al final de la oclusión (frame 59): {abs(est[59, 0] - true[59, 0]):.1f}px")

Prueba a cambiar la trayectoria a una curva (`true[:, 1] = 300 + 50*np.sin(np.arange(T)/10)`): durante la
oclusión el Kalman de velocidad constante se va recto y se equivoca → justo el problema que ataca **OC-SORT**.

## 3. Asociación: el algoritmo húngaro

En cada frame tenemos N tracks (con su caja predicha) y M detecciones. Construimos una **matriz de costes**
`C[i, j]` (típicamente `1 − IoU(pred_i, det_j)`, opcionalmente mezclada con distancia de apariencia) y
buscamos la asignación uno-a-uno de **coste total mínimo**. El **algoritmo húngaro** lo resuelve en O(n³)
(`scipy.optimize.linear_sum_assignment`; también `lap.lapjv`, más rápido, que usa Ultralytics).

Después se aplica una **puerta** (*gating*): parejas con coste > umbral (IoU < 0.3, p. ej.) se descartan aunque
el húngaro las haya elegido. Tracks sin pareja → "perdidos este frame"; detecciones sin pareja → candidatos
a track nuevo.

¿Por qué no un *greedy* (cada track coge su mejor detección)? Porque puede ser globalmente peor:

In [ ]:
C = np.array([[0.1, 0.2],
              [0.15, 0.9]])     # track 0 prefiere det 0, pero track 1 solo encaja bien con det 0
r, c = linear_sum_assignment(C)
print("húngaro:", list(zip(r, c)), "coste", C[r, c].sum())
print("greedy (t0→d0, t1→d1): coste", C[0, 0] + C[1, 1])

**Mahalanobis gating (DeepSORT)**: en vez de IoU, mide la distancia de la detección a la predicción en unidades
de la incertidumbre del Kalman: $d^2 = y^\top S^{-1} y$. Si un track lleva tiempo perdido, $S$ es grande y
acepta detecciones más lejanas. Se descartan parejas con $d^2 > \chi^2_{0.95}(4) = 9.49$.

## 4. ✍️ EJERCICIO: SORT desde cero

SORT (Bewley et al., 2016) = **Kalman + húngaro sobre IoU + reglas de ciclo de vida**:
- Por frame: `predict` de todos los tracks → matriz `1 − IoU` → húngaro → puerta `IoU ≥ iou_thr`.
- Emparejados → `update`. Detecciones sin pareja → **track nuevo** (tentativo).
- Un track se **confirma** tras `min_hits` emparejamientos seguidos (filtra FPs aislados del detector).
- Un track sin emparejar más de `max_age` frames se **borra**.
- Se devuelven solo tracks confirmados emparejados en este frame.

Completa `SimpleSORT.update(dets)` (dets: `(N, 5)` = x1 y1 x2 y2 score). Usa `MyKalmanBox` (convierte
xyxy ↔ cxcywh).

In [ ]:
def xyxy2cxcywh(b): return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, b[2] - b[0], b[3] - b[1]])
def cxcywh2xyxy(s): return np.array([s[0] - s[2] / 2, s[1] - s[3] / 2, s[0] + s[2] / 2, s[1] + s[3] / 2])

class SimpleSORT:
    def __init__(self, iou_thr=0.3, max_age=30, min_hits=3, det_thr=0.5):
        self.iou_thr, self.max_age, self.min_hits, self.det_thr = iou_thr, max_age, min_hits, det_thr
        self.tracks = []     # dicts: {"kf", "id", "hits", "lost"}
        self.next_id = 1

    def update(self, dets):
        # TU CÓDIGO AQUÍ → devuelve array (M, 5): id, x1, y1, x2, y2
        raise NotImplementedError

In [ ]:
# SOLUCIÓN
def _sort_update(self, dets):
    dets = dets[dets[:, 4] >= self.det_thr]
    preds = np.array([cxcywh2xyxy(t["kf"].predict()) for t in self.tracks]).reshape(-1, 4)
    for t in self.tracks:
        t["lost"] += 1
    matched_t, matched_d = set(), set()
    if len(preds) and len(dets):
        iou = iou_matrix(preds, dets[:, :4])
        r, c = linear_sum_assignment(1 - iou)
        for i, j in zip(r, c):
            if iou[i, j] >= self.iou_thr:
                t = self.tracks[i]
                t["kf"].update(xyxy2cxcywh(dets[j, :4]))
                t["hits"] += 1; t["lost"] = 0
                matched_t.add(i); matched_d.add(j)
    for j in range(len(dets)):
        if j not in matched_d:
            self.tracks.append({"kf": MyKalmanBox(xyxy2cxcywh(dets[j, :4])), "id": None, "hits": 1, "lost": 0})
    out = []
    for t in self.tracks:
        if t["id"] is None and t["hits"] >= self.min_hits:
            t["id"] = self.next_id; self.next_id += 1
        if t["id"] is not None and t["lost"] == 0:
            out.append([t["id"], *cxcywh2xyxy(t["kf"].x)])
    self.tracks = [t for t in self.tracks if t["lost"] <= self.max_age and not (t["id"] is None and t["lost"] > 0)]
    return np.array(out).reshape(-1, 5)

SimpleSORT.update = _sort_update

### Escena sintética con ground truth
`make_scene` simula 18 personas con movimiento curvo que se cruzan, se ocluyen (las de delante tapan a las de
detrás), salen y vuelven a entrar del encuadre, y un detector imperfecto: ruido en las cajas, fallos,
falsos positivos, y **scores bajos cuando la persona está parcialmente ocluida** (lo que explota ByteTrack).
También simula un embedding de apariencia por detección (lo usaremos en el notebook 07).

In [ ]:
scene = make_scene(seed=0)
print(f"{scene.n_frames} frames, {len(scene.gt)} cajas GT, {sum(len(d) for d in scene.dets)} detecciones")
fig, axs = plt.subplots(1, 3, figsize=(18, 4))
for ax, f in zip(axs, [50, 150, 250]):
    draw_scene(scene, f, ax=ax)
    d = scene.dets[f - 1]
    for x1, y1, x2, y2, s in d:
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="r" if s < 0.5 else "g", lw=1, ls="--"))
axs[0].set_title("GT gris · det alta verde · det baja roja"); plt.show()

In [ ]:
trk = SimpleSORT()
rows = []
for f in range(scene.n_frames):
    for r in trk.update(scene.dets[f]):
        rows.append([f + 1, *r])
my_sort = np.array(rows)
res = [evaluate(scene.gt, scene.gt, "oráculo (GT)"), evaluate(scene.gt, my_sort, "mi SORT")]
pd.DataFrame(res).round(3)

## 5. Métricas de tracking

Todas parten de emparejar, en cada frame, predicciones con GT (IoU ≥ umbral).

**MOTA** (CLEAR MOT, 2008):
$$ MOTA = 1 - \frac{FN + FP + IDSW}{GT} $$
- Dominada por la **detección** (FN y FP son muchos más que IDSW). Un ID switch cuenta **1** aunque el ID
  erróneo dure 500 frames. Puede ser negativa.
- **IDSW**: un GT emparejado con un ID de predicción distinto al de la última vez.
- **Frag**: veces que una trayectoria GT se interrumpe y se recupera. **MT/ML**: GT seguidos ≥80% / ≤20% del tiempo.

**IDF1** (2016): asigna **globalmente** cada ID de GT a un único ID predicho (húngaro sobre nº de frames
coincidentes) y mide
$$ IDF1 = \frac{2\,IDTP}{2\,IDTP + IDFP + IDFN} $$
→ "¿qué fracción del tiempo cada persona lleva **su** ID?". Es **la métrica de conservación de identidad**.

**HOTA** (2020): $HOTA_\alpha = \sqrt{DetA_\alpha \cdot AssA_\alpha}$, promediada en umbrales α = 0.05…0.95.
- $DetA = TP / (TP + FN + FP)$ → calidad de detección.
- $AssA$ = media sobre TPs de $\frac{TPA}{TPA + FNA + FPA}$ → para cada pareja (GT, pred): de todo lo que
  esa persona y ese ID existen, cuánto coinciden.
- Equilibra detección y asociación, y se puede **descomponer** para diagnosticar (¿falla el detector o el
  tracker?). Es la principal en MOT20, DanceTrack, SportsMOT y KITTI.

Implementación de referencia: **TrackEval** (oficial de MOTChallenge). Aquí usamos `vision_utils/mot_metrics.py`
(implementado desde cero; ábrelo y léelo, son ~130 líneas).

### Ejemplo donde MOTA e IDF1 discrepan

In [ ]:
gt1 = np.array([[f, 1, 100 + f, 100, 150 + f, 250] for f in range(1, 101)], float)
pred_switch = gt1.copy(); pred_switch[50:, 1] = 2          # ID cambia a mitad
pred_miss = gt1.copy(); pred_miss = pred_miss[~np.isin(pred_miss[:, 0], range(40, 50))]   # pierde 10 frames, mismo ID
pd.DataFrame([evaluate(gt1, pred_switch, "1 ID switch a mitad"), evaluate(gt1, pred_miss, "10 frames perdidos")])[
    ["tracker", "MOTA", "IDF1", "HOTA", "DetA", "AssA", "IDSW", "FN"]].round(3)

MOTA prefiere el tracker con el switch (0.99 vs 0.90); IDF1 y HOTA prefieren el que conserva la identidad.
**Para este puesto (identidades) reporta IDF1 y HOTA, y usa MOTA como métrica de detección.**

## 6. ByteTrack

Observación de ByteTrack (2021): las detecciones de **baja confianza** no son basura; muchas son **personas
parcialmente ocluidas**. Tirarlas (como hace SORT con un umbral) rompe los tracks justo en las oclusiones.

**Asociación en dos etapas**:
1. Detecciones **altas** (score ≥ `track_high_thresh`) vs **todos** los tracks (IoU + húngaro).
2. Detecciones **bajas** (`track_low_thresh` ≤ score < alto) vs los tracks que siguen **sin emparejar**, solo con
   IoU (la apariencia de una caja ocluida no es fiable).
3. Las bajas sobrantes **se descartan** (no crean tracks → no hay FPs nuevos); los tracks nuevos solo nacen de
   detecciones altas con score ≥ `new_track_thresh`.
4. Tracks perdidos se mantienen `track_buffer` frames (30) por si vuelven.

Sin red de apariencia → **muy rápido**, y muy buen MOTA/HOTA. Es el baseline por defecto de la industria.

In [ ]:
results = [evaluate(scene.gt, run_tracker(scene, mode="sort"), "SORT"),
           evaluate(scene.gt, run_tracker(scene, mode="byte"), "ByteTrack")]
pd.DataFrame(results).round(3)

### Ajuste de hiperparámetros del tracker
Barrer y medir es lo que harías en el trabajo ("evaluación y comparación de estrategias de tracking"). **Ojo:**
en un caso real, ajusta en un split de validación y reporta en otro de test, o estarás sobreajustando el tracker.

In [ ]:
rows = []
for max_age in [5, 15, 30, 60]:
    for high in [0.4, 0.5, 0.6]:
        r = evaluate(scene.gt, run_tracker(scene, mode="byte", max_age=max_age, high_thr=high), "")
        rows.append({"max_age": max_age, "high_thr": high, "HOTA": r["HOTA"], "IDF1": r["IDF1"], "IDSW": r["IDSW"], "MOTA": r["MOTA"]})
sweep = pd.DataFrame(rows)
sweep.pivot(index="max_age", columns="high_thr", values="IDF1").round(3)

## 7. Análisis de ID switches

Para mejorar hay que entender **cuándo y por qué** cambian las identidades. Localizamos cada switch y miramos
qué pasaba: ¿cruce con otra persona? ¿oclusión larga? ¿detección perdida?

In [ ]:
def find_id_switches(gt, pr, thr=0.5):
    """Lista de switches: frame, gt_id, id anterior, id nuevo, frames sin emparejar y máximo solape con otra
    persona desde la última vez que se emparejó (si es alto → hubo cruce/oclusión)."""
    last, maxov, out = {}, {}, []
    for f in np.unique(gt[:, 0]).astype(int):
        g = gt[gt[:, 0] == f]; p = pr[pr[:, 0] == f]
        Mg = iou_matrix(g[:, 2:6], g[:, 2:6]); np.fill_diagonal(Mg, 0)
        for i, gid in enumerate(g[:, 1].astype(int)):
            maxov[gid] = max(maxov.get(gid, 0), Mg[i].max() if len(g) > 1 else 0)
        if not len(p): continue
        M = iou_matrix(g[:, 2:6], p[:, 2:6])
        r, c = linear_sum_assignment(-M)
        for i, j in zip(r, c):
            if M[i, j] < thr: continue
            gid, pid = int(g[i, 1]), int(p[j, 1])
            if gid in last and last[gid][0] != pid:
                out.append(dict(frame=f, gt_id=gid, antes=last[gid][0], despues=pid,
                                frames_sin_ver=f - last[gid][1], solape_con_otro=round(maxov[gid], 2)))
            last[gid] = (pid, f)
            maxov[gid] = Mg[i].max() if len(g) > 1 else 0
    return pd.DataFrame(out)

byte = run_tracker(scene, mode="byte")
sw = find_id_switches(scene.gt, byte)
sw

In [ ]:
# Clasificación de las causas
if len(sw):
    causa = np.where(sw.frames_sin_ver > 30, "desaparición > max_age",
             np.where(sw.solape_con_otro > 0.2, "cruce / oclusión", "otro (fallo detección, movimiento)"))
    print(pd.Series(causa).value_counts())

In [ ]:
# Visualizar un switch: trayectorias GT y predichas alrededor del frame
if len(sw):
    s = sw.iloc[0]
    f0, f1 = s.frame - 40, s.frame + 40
    plt.figure(figsize=(8, 6))
    g = scene.gt[(scene.gt[:, 1] == s.gt_id) & (scene.gt[:, 0] >= f0) & (scene.gt[:, 0] <= f1)]
    plt.plot((g[:, 2] + g[:, 4]) / 2, g[:, 5], "k-", lw=4, alpha=0.3, label=f"GT {s.gt_id}")
    for pid in [s.antes, s.despues]:
        p = byte[(byte[:, 1] == pid) & (byte[:, 0] >= f0) & (byte[:, 0] <= f1)]
        plt.plot((p[:, 2] + p[:, 4]) / 2, p[:, 5], ".-", label=f"track {pid}")
    plt.gca().invert_yaxis(); plt.legend(); plt.title(f"ID switch en frame {s.frame} (pies de las cajas)"); plt.show()

**Recetario para reducir ID switches** (memorízalo, es pregunta segura):
| Causa | Remedio |
|---|---|
| Cruces entre personas | apariencia (ReID) en la asociación, OC-SORT (consistencia de dirección), mejor detector en solapes |
| Oclusión larga / salida del encuadre | `track_buffer` mayor + **ReID para re-identificar**; *stitching* offline de tracklets |
| Movimiento no lineal | OC-SORT (ORU/OCM), Kalman con más ruido de proceso, frame rate mayor |
| Cámara en movimiento | **GMC** (compensación de movimiento global, BoT-SORT) |
| Detecciones perdidas/ocluidas | ByteTrack (usar las de baja confianza), bajar umbral del detector, mejor detector |
| Escena casi vista desde arriba | asociar en coordenadas del campo (homografía, notebook 08) en vez de en píxeles |

## 8. ✍️ EJERCICIO: post-procesado offline — interpolación de huecos

Si el análisis es **offline** (procesar el partido después), se puede mejorar mucho:
- **Interpolación lineal** de los huecos cortos de cada track (frames donde el track estaba perdido y luego
  volvió con el mismo ID) → reduce FN y Frag. StrongSORT propone **GSI** (interpolación + suavizado gaussiano).
- **Stitching** de tracklets por apariencia y continuidad espacio-temporal (notebook 07).

Implementa `interpolate_tracks(mot, max_gap=20)`: para cada ID, si entre dos frames consecutivos del track hay un
hueco ≤ `max_gap`, añade filas interpolando linealmente las 4 coordenadas.

In [ ]:
def interpolate_tracks(mot, max_gap=20):
    # TU CÓDIGO AQUÍ → devuelve array (K', 6+) ordenado por frame
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def interpolate_tracks(mot, max_gap=20):
    out = [mot[:, :6]]
    for tid in np.unique(mot[:, 1]):
        t = mot[mot[:, 1] == tid]
        t = t[np.argsort(t[:, 0])]
        for a, b in zip(t[:-1], t[1:]):
            gap = int(b[0] - a[0])
            if 1 < gap <= max_gap:
                for k in range(1, gap):
                    w = k / gap
                    out.append(np.r_[a[0] + k, tid, (1 - w) * a[2:6] + w * b[2:6]][None])
    res = np.concatenate(out)
    return res[np.lexsort((res[:, 1], res[:, 0]))]

In [ ]:
byte_interp = interpolate_tracks(byte)
pd.DataFrame([evaluate(scene.gt, byte, "ByteTrack"), evaluate(scene.gt, byte_interp, "ByteTrack + interpolación")]).round(3)

## 9. Mapa de trackers (una o dos frases cada uno — apréndetelas)

| Tracker | Año | Idea clave | Cuándo |
|---|---|---|---|
| **SORT** | 2016 | Kalman + húngaro sobre IoU | baseline, rapidísimo |
| **DeepSORT** | 2017 | + **apariencia** (CNN ReID de 128-d, distancia coseno con galería de últimos embeddings), **Mahalanobis gating**, **matching cascade** (prioriza tracks vistos recientemente) | oclusiones largas, más coste |
| **JDE / FairMOT** | 2020 | detector y ReID en **una sola red** (cabeza de embedding) | menos coste que detector + ReID separado |
| **ByteTrack** | 2021 | asociación en **dos etapas** usando detecciones de baja confianza | baseline industrial, sin apariencia |
| **OC-SORT** | 2022 | "*observation-centric*": **ORU** (al recuperar un track tras perderlo, re-ajusta el Kalman con una trayectoria virtual entre la última y la nueva observación → corrige el error acumulado), **OCM** (añade al coste la consistencia de **dirección** de movimiento), **OCR** (segunda pasada con la última observación) | **movimiento no lineal**, oclusiones (DanceTrack, deporte) |
| **BoT-SORT** | 2022 | ByteTrack + **GMC** (compensación de movimiento de cámara con flujo óptico/ORB + RANSAC), Kalman con w,h, fusión **IoU + ReID** | cámara móvil, top en MOT17 |
| **StrongSORT** | 2022 | DeepSORT modernizado (mejor ReID, EMA de embeddings, GMC) + **AFLink** (enlace offline de tracklets) + **GSI** (interpolación gaussiana) | offline, máxima precisión |
| **Deep OC-SORT** | 2023 | OC-SORT + apariencia adaptativa + GMC | |
| **MOTR / MOTRv2 / MOTIP** | 2022+ | **end-to-end con transformers**: *track queries* que se propagan entre frames | investigación; caros |
| **SAM2 / SAMURAI** | 2024 | trackers de máscaras con memoria (VOS) | pocos objetos, offline |

Ultralytics incluye `bytetrack`, `botsort` (con `with_reid` opcional), `ocsort`, `deepocsort` y otros. Para
comparar trackers fuera de Ultralytics: la librería **BoxMOT** (DeepOCSORT, StrongSORT, BoT-SORT, etc. con
modelos ReID intercambiables).

**Benchmarks**: MOT17 (peatones), MOT20 (multitudes densas), **DanceTrack** (misma ropa, movimiento no lineal →
la apariencia no basta), **SportsMOT** (baloncesto, fútbol, voleibol), **SoccerNet Tracking/Game State**
(fútbol: tracking + identificación de dorsal + posición en el campo). Si el puesto es deportivo, menciona
SportsMOT y SoccerNet.

## 10. Tracking real con Ultralytics

`model.track(...)` = detector + tracker. Detalles importantes:
- `persist=True` cuando llamas frame a frame (si no, el tracker se reinicia en cada llamada).
- `conf` del detector: déjalo **bajo** (≈0.1) para que ByteTrack/BoT-SORT reciban las detecciones bajas.
- Configura el tracker con un YAML propio (copia `bytetrack.yaml` y cambia `track_buffer`, umbrales...).
- `r.boxes.id` puede ser `None` si en ese frame no hay tracks confirmados.

In [ ]:
import cv2, torch
from collections import defaultdict
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
VIDEO = DATA / "people.mp4"

def track_video(tracker_cfg, max_frames=250, imgsz=960):
    model = YOLO(str(DATA / "yolo11n.pt"))
    cap = cv2.VideoCapture(str(VIDEO))
    rows, t0, n = [], time.perf_counter(), 0
    while n < max_frames:
        ok, frame = cap.read()
        if not ok: break
        n += 1
        r = model.track(frame, persist=True, tracker=tracker_cfg, classes=[0], conf=0.1, imgsz=imgsz,
                        device=DEVICE, verbose=False)[0]
        if r.boxes.id is not None:
            for tid, b, s in zip(r.boxes.id.int().tolist(), r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                rows.append([n, tid, *b, s])
    cap.release()
    fps = n / (time.perf_counter() - t0)
    return np.array(rows), fps

real = {}
for cfg in ["bytetrack.yaml", "botsort.yaml", "ocsort.yaml"]:
    try:
        real[cfg], fps = track_video(cfg)
        lens = pd.Series(real[cfg][:, 1]).value_counts()
        print(f"{cfg:15s} {fps:5.1f} FPS | IDs únicos: {len(lens):3d} | tracks > 50 frames: {(lens > 50).sum():3d} | "
              f"longitud media: {lens.mean():.0f} frames")
    except Exception as e:
        print(cfg, "no disponible en esta versión:", e)

**Sin GT, ¿cómo comparo trackers en un vídeo real?** Métricas *proxy*: nº de IDs únicos frente al nº real de
personas (muchos más IDs = fragmentación/switches), longitud de los tracks, nº de tracks cortos. Y siempre
**mirar el vídeo**. Para una evaluación seria hay que anotar GT de tracking (CVAT, con interpolación) en unos
cuantos clips representativos y usar TrackEval.

In [ ]:
# Vídeo anotado con trayectorias (rastro de los últimos 30 frames)
mot = real["bytetrack.yaml"]
cap = cv2.VideoCapture(str(VIDEO))
W, H = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(str(DATA / "06_tracking.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS), (W, H))
trail = defaultdict(list)
colors = {}
for f in range(1, int(mot[:, 0].max()) + 1):
    ok, frame = cap.read()
    if not ok: break
    for _, tid, x1, y1, x2, y2, s in mot[mot[:, 0] == f]:
        tid = int(tid)
        c = colors.setdefault(tid, tuple(int(v) for v in np.random.default_rng(tid).integers(50, 255, 3)))
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
        cv2.putText(frame, str(tid), (int(x1), int(y1) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c, 2)
        trail[tid].append((int((x1 + x2) / 2), int(y2)))
        for p, q in zip(trail[tid][-30:-1], trail[tid][-29:]):
            cv2.line(frame, p, q, c, 2)
    out.write(frame)
cap.release(); out.release()
plt.figure(figsize=(14, 8)); plt.imshow(frame[:, :, ::-1]); plt.axis("off"); plt.title("ByteTrack + rastro de 30 frames");
print("Vídeo guardado en", DATA / "06_tracking.mp4")